In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

import numpy as np
import pandas as pd

In [3]:
import anndata
import scanpy as sc
import celloracle as co

/picb/lihonglab/chenxufeng/miniconda3/envs/celloracle/lib/python3.8/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/picb/lihonglab/chenxufeng/miniconda3/envs/celloracle/lib/python3.8/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def dna_to_t

In [4]:
import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt

## Configurations

In [5]:
adata = sc.read("/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn100_noise_tree1_1000_cells110_genes_sigma0.1_7/res.h5ad")

In [6]:
adata

AnnData object with n_obs × n_vars = 1000 × 110
    obs: 'cell_id', 'pop', 'depth', 'batch', 'n_counts', 'leiden', 'dpt_pseudotime', 'palantir_pseudotime'
    var: 'is_reg', 'is_target'
    uns: 'diffmap_evals', 'iroot', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs', 'base_grn'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [7]:
base_grn = adata.varm["base_grn"].copy()

In [8]:
adata.obs["leiden"] = "1"

In [9]:
adata.X.A

array([[1.6254034 , 0.05550044, 0.20561314, ..., 0.04191143, 0.        ,
        0.20561314],
       [0.28096858, 0.06539987, 0.16188341, ..., 0.        , 0.        ,
        0.12678389],
       [0.69717133, 0.33877373, 0.29685643, ..., 0.05600567, 0.        ,
        0.39852864],
       ...,
       [0.596284  , 0.0880629 , 0.3695844 , ..., 0.025962  , 0.        ,
        0.19096375],
       [0.33675823, 0.15429369, 0.5882315 , ..., 0.        , 0.        ,
        0.55046976],
       [0.59768265, 0.15679269, 0.25712737, ..., 0.14351228, 0.        ,
        0.39095706]], dtype=float32)

In [10]:
adata.X = adata.layers["counts"].A.copy()

In [11]:
adata.X

array([[286.,   4.,  16., ...,   3.,   0.,  16.],
       [ 24.,   5.,  13., ...,   0.,   0.,  10.],
       [ 35.,  14.,  12., ...,   2.,   0.,  17.],
       ...,
       [ 62.,   7.,  34., ...,   2.,   0.,  16.],
       [ 12.,   5.,  24., ...,   0.,   0.,  22.],
       [ 53.,  11.,  19., ...,  10.,   0.,  31.]], dtype=float32)

In [12]:
sc.pp.normalize_per_cell(adata, counts_per_cell_after = 100)

In [13]:
adata.X

array([[4.080468  , 0.05706948, 0.22827794, ..., 0.04280211, 0.        ,
        0.22827794],
       [0.324412  , 0.06758583, 0.17572317, ..., 0.        , 0.        ,
        0.13517167],
       [1.0080645 , 0.40322578, 0.34562212, ..., 0.05760368, 0.        ,
        0.48963133],
       ...,
       [0.8153603 , 0.09205681, 0.4471331 , ..., 0.02630195, 0.        ,
        0.21041557],
       [0.4004004 , 0.1668335 , 0.8008008 , ..., 0.        , 0.        ,
        0.73406744],
       [0.8179012 , 0.16975307, 0.29320985, ..., 0.15432099, 0.        ,
        0.47839504]], dtype=float32)

In [14]:
adata.layers["raw_count"] = adata.X.copy()

In [17]:
oracle = co.Oracle()
# In this notebook, we use the unscaled mRNA count for the input of Oracle object.
# Instantiate Oracle object.
oracle.import_anndata_as_raw_count(adata=adata, cluster_column_name="leiden", embedding_name="X_pca", transform="log2")


110 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.
(109, 1000)


In [235]:
oracle

Oracle object

Meta data
    celloracle version used for instantiation: 0.16.0
    n_cells: 1000
    n_genes: 1250
    cluster_name: leiden
    dimensional_reduction_name: X_pca
    n_target_genes_in_TFdict: 0 genes
    n_regulatory_in_TFdict: 0 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 0 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 0 genes
    k_for_knn_imputation: NA
Status
    Gene expression matrix: Ready
    BaseGRN: Not imported
    PCA calculation: Not finished
    Knn imputation: Not finished
    GRN calculation for simulation: Not finished

In [80]:
# Creat TFinfo_dictionary, each key is a target gene, the value is the list of TF connecting to the target gene in baseGRN
TF_to_TG_dictionary = {}
for tf_id, tf in enumerate(base_grn.columns.values):
    TF_to_TG_dictionary[tf] = [x for x in base_grn.index.values[base_grn.loc[:, tf] != 0]] 

TG_to_TF_dictionary = co.utility.inverse_dictionary(TF_to_TG_dictionary)

# You can load TF info dataframe with the following code.
oracle.import_TF_data(TFdict=TG_to_TF_dictionary)

  0%|          | 0/90 [00:00<?, ?it/s]

Total number of TF was 6. Although we can go to the GRN calculation with this data, but the TF number is small.


In [81]:
oracle

Oracle object

Meta data
    celloracle version used for instantiation: 0.16.0
    n_cells: 1000
    n_genes: 110
    cluster_name: leiden
    dimensional_reduction_name: X_pca
    n_target_genes_in_TFdict: 90 genes
    n_regulatory_in_TFdict: 6 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 6 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 90 genes
    k_for_knn_imputation: NA
Status
    Gene expression matrix: Ready
    BaseGRN: Ready
    PCA calculation: Not finished
    Knn imputation: Not finished
    GRN calculation for simulation: Not finished

In [82]:
# knn imputation
# Perform PCA
oracle.perform_PCA()

# Select important PCs
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
plt.show()
print(n_comps)
n_comps = min(n_comps, 50)

n_cell = oracle.adata.shape[0]
k = int(0.025*n_cell)
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8, b_maxl=k*4, n_jobs=4)

54


In [83]:
links = oracle.get_links(cluster_name_for_GRN_unit="leiden", alpha=10, verbose_level=10)

  0%|          | 0/1 [00:00<?, ?it/s]

Inferring GRN for 1...


  0%|          | 0/90 [00:00<?, ?it/s]

In [53]:
links

In [54]:
raw_edge_df = links.links_dict["1"]

In [55]:
raw_edge_df

,source,target,coef_mean,coef_abs,p,-logp
0,gene803,gene1,-0.000580,0.000580,1.099063e-05,4.958977
1,gene684,gene1,0.000206,0.000206,1.495152e-09,8.825315
2,gene231,gene1,0.000354,0.000354,3.765850e-03,2.424137
3,gene803,gene10,0.059695,0.059695,4.683301e-10,9.329448
4,gene2,gene10,0.008050,0.008050,3.360035e-02,1.473656
...,...,...,...,...,...,...
1778,gene803,gene996,0.010205,0.010205,9.016290e-16,15.044972
1779,gene684,gene996,-0.001418,0.001418,7.434000e-07,6.128777
1780,gene406,gene998,0.036336,0.036336,2.994331e-32,31.523700
1781,gene803,gene999,-0.023117,0.023117,7.920663e-14,13.101238


In [56]:
links.filter_links(p=0.001)

In [57]:
edge_df = links.links_dict["1"]
edge_df = edge_df.loc[:, ['source', 'target', 'coef_abs']]
edge_df.columns = ['TF', 'Target', 'Score']


In [58]:
edge_df

,TF,Target,Score
0,gene803,gene1,0.000580
1,gene684,gene1,0.000206
2,gene231,gene1,0.000354
3,gene803,gene10,0.059695
4,gene2,gene10,0.008050
...,...,...,...
1778,gene803,gene996,0.010205
1779,gene684,gene996,0.001418
1780,gene406,gene998,0.036336
1781,gene803,gene999,0.023117


In [60]:
edge_df.to_csv("/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn1139_noise_tree1_1000_cells1139_genes_sigma0.1_1/celloracle_edge_df.csv", index=False)

## Batch

In [27]:
def run_celloracle(path, p = 0, output_dir = "/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/net/CellOracle/"):
    
    ds_name = path.split("/")[-2]
    adata = sc.read(os.path.join(path, "res.h5ad"))
    
    if p == 0:
        base_grn = adata.varm["base_grn"].copy()
    elif p > 0:
        base_grn = adata.varm[f"noisy_grn_{p}"].copy()
        ds_name = ds_name + f"_p{p}"
    else:
        raise ValueError("p must be >= 0")
    
    adata.obs["leiden"] = "1"
    
    adata.X = adata.layers["counts"].A.copy()
    
    sc.pp.normalize_per_cell(adata, counts_per_cell_after = 100)
    
    oracle = co.Oracle()
    oracle.import_anndata_as_raw_count(adata=adata, cluster_column_name="leiden", embedding_name="X_pca", transform="log2")
    
    # Creat TFinfo_dictionary, each key is a target gene, the value is the list of TF connecting to the target gene in baseGRN
    TF_to_TG_dictionary = {}
    for tf_id, tf in enumerate(base_grn.columns.values):
        TF_to_TG_dictionary[tf] = [x for x in base_grn.index.values[base_grn.loc[:, tf] != 0]] 

    TG_to_TF_dictionary = co.utility.inverse_dictionary(TF_to_TG_dictionary)

    # You can load TF info dataframe with the following code.
    oracle.import_TF_data(TFdict=TG_to_TF_dictionary)
    
    # knn imputation
    # Perform PCA
    oracle.perform_PCA()

    # Select important PCs
    plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
    n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
    plt.axvline(n_comps, c="k")
    plt.show()
    print(n_comps)
    n_comps = min(n_comps, 50)

    n_cell = oracle.adata.shape[0]
    k = int(0.025*n_cell)
    oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8, b_maxl=k*4, n_jobs=4)
    
    links = oracle.get_links(cluster_name_for_GRN_unit="leiden", alpha=10, verbose_level=10)
    
    links.filter_links(p=0.001)
    
    edge_df = links.links_dict["1"]
    edge_df = edge_df.loc[:, ['source', 'target', 'coef_abs']]
    edge_df.columns = ['TF', 'Target', 'Score']

    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    edge_df.to_csv(os.path.join(output_dir, f"{ds_name}.csv"), index=None)

In [28]:
run_celloracle("/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn1139_noise_tree1_1000_cells1139_genes_sigma0.1_1/", p=0.01)

(1250, 1000)


  0%|          | 0/1130 [00:00<?, ?it/s]

Total number of TF was 6. Although we can go to the GRN calculation with this data, but the TF number is small.
35


  0%|          | 0/1 [00:00<?, ?it/s]

Inferring GRN for 1...


  0%|          | 0/1130 [00:00<?, ?it/s]

In [24]:
ds_dir = "/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/"
datasets = [ds_dir + name + "/" for name in os.listdir(ds_dir) if os.path.isdir(os.path.join(ds_dir, name))]

In [25]:
datasets[24]

'/home/chenxufeng/picb_cxf/Data/scMultiSim/bench_grn/datasets/grn100_noise_tree1_1000_cells110_genes_sigma0.1_7/'

In [ ]:
from tqdm.auto import tqdm
for dataset in tqdm(datasets, desc=f"Processing datasets: ", unit=f"{datasets[0].split('/')[-2]}"):
    run_celloracle(dataset)

In [ ]:
from tqdm.auto import tqdm
for dataset in tqdm(datasets):
    for p in [0.05]:
        run_celloracle(dataset, p=p)